> # ⚠️ ARCHIVED — DO NOT CITE ANY NUMBER FROM THIS NOTEBOOK
>
> This notebook is from the project's first generation (2026-08-11). It is kept
> to show the methodological path, **not** as evidence.
>
> - Its stored outputs have been **stripped**, deliberately, so no figure or table
>   here can be mistaken for a current result. Git history retains them.
> - It reads data paths and episode identifiers that **no longer exist**, so it
>   cannot be re-executed to regenerate them.
> - Where it uses the Grand Ouest reference, note that the reference has since been
>   re-resolved onto a new episode reconstruction, and the linkage methods,
>   thresholds, and splits all changed afterwards. Same source data, different
>   everything else.
>
> Current evidence lives in `notebooks/10`–`14`, the `*.md` reports at the
> repository root, and `reports/boamp_methodology_chapter.pdf`.


# BOAMP Renewal Candidate Pair Generation

## tl;dr

Executed successfully. The benchmark-anchored candidate space contains `8,095` candidate pairs across all `120` reference anchors, including all `94` primary evaluation anchors. This is a bounded first baseline candidate table for comparing linkage methods before any full-corpus linkage application.

## Context & Methods

Candidate generation is not final linkage. It creates plausible later notices for each anchor using temporal, buyer, CPV/theme, geography, and text evidence. The table is the common input for all baseline algorithms.


In [ ]:
from __future__ import annotations

import ast
import json
import math
import re
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data/processed/boamp_grand_ouest/notices_engineered.parquet").exists():
            return candidate
    return Path.cwd()

PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data/processed/boamp_grand_ouest"
NOTICES_PATH = PROCESSED_DIR / "notices_engineered.parquet"
ANCHOR_PATH = PROCESSED_DIR / "reference_anchor_episodes.parquet"
OUTPUT_PATH = PROCESSED_DIR / "renewal_candidate_pairs.parquet"
SUMMARY_PATH = PROCESSED_DIR / "candidate_pair_generation_summary.json"

MIN_GAP_DAYS = 90
MAX_GAP_DAYS = 8 * 365
MAX_CANDIDATES_PER_ANCHOR = 75
BUYER_SIMILARITY_MIN = 0.72
TEXT_KEEP_MIN = 0.03

print(PROJECT_ROOT)


## Data

### 1. Load Engineered Notices And Reference Anchors


In [ ]:
def parse_json_list(value) -> list:
    if value is None or pd.isna(value):
        return []
    text = str(value).strip()
    if text == "":
        return []
    try:
        parsed = json.loads(text)
    except Exception:
        return []
    return parsed if isinstance(parsed, list) else [parsed]


def normalize_text(value) -> str:
    if value is None or pd.isna(value):
        return ""
    text = "".join(ch for ch in unicodedata.normalize("NFKD", str(value).lower()) if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_buyer_for_blocking(value) -> str:
    text = normalize_text(value)
    text = re.sub(r"\b(commune de|ville de|mairie de|departement de|departement|region|conseil regional|conseil departemental|ct[eé] de cnes|communaute de communes|communaute d agglomeration|metropole)\b", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def buyer_similarity(a: str, b: str) -> float:
    if not a or not b:
        return 0.0
    if a == b:
        return 1.0
    return SequenceMatcher(None, a, b).ratio()


def buyer_tokens(value: str) -> set[str]:
    return {token for token in str(value).split() if len(token) >= 4 and not token.isdigit()}


def jaccard(left: set, right: set) -> float:
    if not left or not right:
        return 0.0
    return len(left & right) / len(left | right)


def duration_gap_score(candidate_date, expected_end_date, time_gap_days) -> float:
    if pd.notna(expected_end_date):
        diff = abs((candidate_date - expected_end_date).days)
        return max(0.0, 1.0 - diff / 365.0)
    target = 4 * 365
    return max(0.0, 1.0 - abs(time_gap_days - target) / (4 * 365))

notices = pd.read_parquet(NOTICES_PATH, columns=[
    "idweb", "dateparution", "buyer_name_raw", "buyer_name_normalized", "buyer_key_best", "buyer_key_source",
    "objet_normalized", "grand_ouest_department", "grand_ouest_region", "cpv_codes_json", "cpv_prefix2_json",
    "primary_cpv_prefix2", "technology_segment_rule_based", "is_digital_candidate", "has_cpv", "has_buyer_identifier",
    "duration_missing", "declared_duration_months", "expected_end_date", "amount_best", "usable_for_linkage", "url_avis"
])
notices["dateparution_dt"] = pd.to_datetime(notices["dateparution"], errors="coerce")
notices["candidate_text"] = notices["objet_normalized"].fillna("").astype(str)
notices["buyer_name_normalized"] = notices["buyer_name_normalized"].fillna("").astype(str)
notices["buyer_key_best"] = notices["buyer_key_best"].fillna("").astype(str)
notices["cpv_prefix_set"] = notices["cpv_prefix2_json"].map(lambda value: set(parse_json_list(value)))
notices = notices[notices["dateparution_dt"].notna()].reset_index(drop=True).copy()
notices["row_pos"] = np.arange(len(notices))
notices["buyer_tokens"] = notices["buyer_name_normalized"].map(buyer_tokens)

buyer_key_index = {key: set(group["row_pos"].tolist()) for key, group in notices.groupby("buyer_key_best") if str(key).strip()}
buyer_name_index = {key: set(group["row_pos"].tolist()) for key, group in notices.groupby("buyer_name_normalized") if str(key).strip()}
buyer_token_index = defaultdict(set)
for row in notices[["row_pos", "buyer_tokens"]].itertuples(index=False):
    for token in row.buyer_tokens:
        buyer_token_index[token].add(int(row.row_pos))

anchors = pd.read_parquet(ANCHOR_PATH)
anchors["anchor_award_notice_date_dt"] = pd.to_datetime(anchors["anchor_award_notice_date"], errors="coerce")
anchors["anchor_expected_end_date_dt"] = pd.to_datetime(anchors["anchor_expected_end_date"].replace("", pd.NA), errors="coerce")
anchors["anchor_notice_ids"] = anchors["anchor_notice_ids_list_json"].map(parse_json_list)
anchors["anchor_cpv_prefix_set"] = anchors["anchor_cpv_codes_list_json"].map(lambda value: {str(c)[:2] for c in parse_json_list(value) if str(c)})
anchors["anchor_buyer_name_normalized"] = anchors["anchor_buyer_name"].map(normalize_buyer_for_blocking)
anchors["anchor_text_normalized"] = anchors["anchor_text"].map(normalize_text)
anchors["anchor_siren_clean"] = anchors["anchor_buyer_siren"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
print(notices.shape, anchors.shape)


### 2. Generate Candidate Pairs For Benchmark Anchors


In [ ]:

def local_tfidf_scores(anchor_text: str, candidate_texts: pd.Series, analyzer="word") -> np.ndarray:
    if candidate_texts.empty:
        return np.array([])
    corpus = [anchor_text] + candidate_texts.fillna("").astype(str).tolist()
    if analyzer == "char":
        vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1)
    else:
        vectorizer = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=1, stop_words=None)
    try:
        matrix = vectorizer.fit_transform(corpus)
        return cosine_similarity(matrix[0], matrix[1:]).ravel()
    except ValueError:
        return np.zeros(len(candidate_texts))

candidate_frames = []
for row in anchors.itertuples(index=False):
    anchor_date = row.anchor_award_notice_date_dt
    if pd.isna(anchor_date):
        continue
    min_date = anchor_date + pd.Timedelta(days=MIN_GAP_DAYS)
    max_date = anchor_date + pd.Timedelta(days=MAX_GAP_DAYS)
    anchor_notice_ids = set(row.anchor_notice_ids)
    anchor_buyer_name = row.anchor_buyer_name_normalized
    anchor_buyer_tokens = buyer_tokens(anchor_buyer_name)
    anchor_siren = row.anchor_siren_clean if row.anchor_siren_clean and row.anchor_siren_clean.lower() != "nan" else ""
    anchor_prefixes = set(row.anchor_cpv_prefix_set)
    anchor_theme = str(row.anchor_theme)
    anchor_theme_prefix = anchor_theme.replace("CPV-", "") if anchor_theme.startswith("CPV-") else ""
    if anchor_theme_prefix:
        anchor_prefixes.add(anchor_theme_prefix)
    anchor_text = row.anchor_text_normalized

    candidate_positions = set()
    if anchor_siren:
        candidate_positions.update(buyer_key_index.get(anchor_siren, set()))
    if anchor_buyer_name:
        candidate_positions.update(buyer_name_index.get(anchor_buyer_name, set()))
    for token in anchor_buyer_tokens:
        candidate_positions.update(buyer_token_index.get(token, set()))

    if not candidate_positions:
        continue

    pool = notices.iloc[sorted(candidate_positions)].copy()
    date_mask = pool["dateparution_dt"].between(min_date, max_date, inclusive="both") & ~pool["idweb"].isin(anchor_notice_ids)
    buyer_pool = pool.loc[date_mask].copy()
    if buyer_pool.empty:
        continue

    buyer_pool["buyer_name_similarity"] = buyer_pool["buyer_name_normalized"].map(lambda value: buyer_similarity(anchor_buyer_name, value)).astype(float)
    buyer_pool["same_buyer_siren"] = buyer_pool["buyer_key_best"].eq(anchor_siren) if anchor_siren else False
    buyer_pool["same_buyer_normalized_name"] = buyer_pool["buyer_name_normalized"].eq(anchor_buyer_name)
    buyer_pool["buyer_token_overlap"] = buyer_pool["buyer_tokens"].map(lambda tokens: jaccard(anchor_buyer_tokens, tokens))
    buyer_mask = buyer_pool["same_buyer_siren"] | buyer_pool["same_buyer_normalized_name"] | buyer_pool["buyer_name_similarity"].ge(BUYER_SIMILARITY_MIN) | buyer_pool["buyer_token_overlap"].ge(0.34)
    buyer_pool = buyer_pool.loc[buyer_mask].copy()
    if buyer_pool.empty:
        continue

    word_scores = local_tfidf_scores(anchor_text, buyer_pool["candidate_text"], analyzer="word")
    char_scores = local_tfidf_scores(anchor_text, buyer_pool["candidate_text"], analyzer="char")
    buyer_pool["word_tfidf_similarity"] = word_scores
    buyer_pool["char_ngram_tfidf_similarity"] = char_scores
    buyer_pool["time_gap_days"] = (buyer_pool["dateparution_dt"] - anchor_date).dt.days.astype(int)
    buyer_pool["time_gap_months"] = (buyer_pool["time_gap_days"] / 30.4375).round(1)
    buyer_pool["buyer_match_type"] = np.select(
        [buyer_pool["same_buyer_siren"], buyer_pool["same_buyer_normalized_name"], buyer_pool["buyer_name_similarity"].ge(BUYER_SIMILARITY_MIN), buyer_pool["buyer_token_overlap"].ge(0.34)],
        ["siren", "normalized_name", "fuzzy_name", "token_overlap"],
        default="none",
    )
    buyer_pool["cpv_overlap_score"] = buyer_pool["cpv_prefix_set"].map(lambda values: jaccard(anchor_prefixes, values))
    buyer_pool["same_cpv_prefix"] = buyer_pool["cpv_overlap_score"].gt(0)
    buyer_pool["same_theme"] = buyer_pool["primary_cpv_prefix2"].fillna("").eq(anchor_theme_prefix)
    buyer_pool["same_region"] = buyer_pool["grand_ouest_region"].astype(str).isin(parse_json_list(row.anchor_regions_list_json))
    expected = row.anchor_expected_end_date_dt
    buyer_pool["duration_gap_score"] = buyer_pool["dateparution_dt"].map(lambda dt: duration_gap_score(dt, expected, int((dt - anchor_date).days)))
    buyer_pool["preliminary_score"] = (
        0.35 * buyer_pool[["buyer_name_similarity", "buyer_token_overlap"]].max(axis=1).clip(0, 1)
        + 0.25 * buyer_pool[["word_tfidf_similarity", "char_ngram_tfidf_similarity"]].max(axis=1).clip(0, 1)
        + 0.20 * buyer_pool["cpv_overlap_score"].clip(0, 1)
        + 0.20 * buyer_pool["duration_gap_score"].clip(0, 1)
    )
    keep_mask = buyer_pool["same_cpv_prefix"] | buyer_pool["same_theme"] | buyer_pool[["word_tfidf_similarity", "char_ngram_tfidf_similarity"]].max(axis=1).ge(TEXT_KEEP_MIN)
    kept = buyer_pool.loc[keep_mask].sort_values("preliminary_score", ascending=False).head(MAX_CANDIDATES_PER_ANCHOR).copy()
    if kept.empty:
        kept = buyer_pool.sort_values("preliminary_score", ascending=False).head(min(10, len(buyer_pool))).copy()

    kept.insert(0, "sample_id", row.sample_id)
    kept.insert(1, "benchmark_split", row.benchmark_split)
    kept.insert(2, "anchor_episode_id", row.anchor_episode_id)
    kept.insert(3, "anchor_notice_ids_json", json.dumps(list(anchor_notice_ids), ensure_ascii=False))
    kept.insert(4, "anchor_award_notice_date", str(row.anchor_award_notice_date))
    kept.insert(5, "anchor_expected_end_date", str(row.anchor_expected_end_date))
    kept.insert(6, "anchor_buyer_name", str(row.anchor_buyer_name))
    kept.insert(7, "anchor_buyer_siren", anchor_siren)
    kept.insert(8, "anchor_theme", anchor_theme)
    kept.insert(9, "anchor_text_normalized", anchor_text)
    kept["candidate_rank_preliminary"] = np.arange(1, len(kept) + 1)
    candidate_frames.append(kept)

pairs = pd.concat(candidate_frames, ignore_index=True) if candidate_frames else pd.DataFrame()
rename_map = {"idweb": "candidate_idweb", "dateparution": "candidate_dateparution", "buyer_name_raw": "candidate_buyer_name", "buyer_name_normalized": "candidate_buyer_name_normalized", "buyer_key_best": "candidate_buyer_key_best", "objet_normalized": "candidate_text_normalized", "grand_ouest_department": "candidate_department", "grand_ouest_region": "candidate_region", "primary_cpv_prefix2": "candidate_primary_cpv_prefix2", "technology_segment_rule_based": "candidate_technology_segment"}
pairs = pairs.rename(columns=rename_map)
columns = [
    "sample_id", "benchmark_split", "anchor_episode_id", "anchor_notice_ids_json", "anchor_award_notice_date", "anchor_expected_end_date",
    "anchor_buyer_name", "anchor_buyer_siren", "anchor_theme", "anchor_text_normalized", "candidate_idweb", "candidate_dateparution",
    "candidate_buyer_name", "candidate_buyer_name_normalized", "candidate_buyer_key_best", "candidate_text_normalized", "candidate_department",
    "candidate_region", "candidate_primary_cpv_prefix2", "candidate_technology_segment", "url_avis", "time_gap_days", "time_gap_months",
    "same_buyer_siren", "same_buyer_normalized_name", "buyer_match_type", "buyer_name_similarity", "buyer_token_overlap", "same_cpv_prefix", "cpv_overlap_score",
    "same_theme", "same_region", "word_tfidf_similarity", "char_ngram_tfidf_similarity", "duration_gap_score", "duration_missing",
    "has_cpv", "has_buyer_identifier", "amount_best", "preliminary_score", "candidate_rank_preliminary"
]
pairs = pairs[[column for column in columns if column in pairs.columns]].copy()
pairs.to_parquet(OUTPUT_PATH, index=False, compression="zstd")
print(f"Wrote {len(pairs):,} candidate pairs for {pairs['sample_id'].nunique() if len(pairs) else 0} anchors")


## Results

### 3. Validate Candidate Space


In [ ]:
pairs = pd.read_parquet(OUTPUT_PATH)
all_anchor_ids = set(anchors["sample_id"])
pair_anchor_ids = set(pairs["sample_id"])
summary = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "input_notices": str(NOTICES_PATH),
    "input_reference_anchors": str(ANCHOR_PATH),
    "output_file": str(OUTPUT_PATH),
    "candidate_rules": {
        "min_gap_days": MIN_GAP_DAYS,
        "max_gap_days": MAX_GAP_DAYS,
        "buyer_similarity_min": BUYER_SIMILARITY_MIN,
        "max_candidates_per_anchor": MAX_CANDIDATES_PER_ANCHOR,
        "benchmark_anchored_first_baseline": True,
    },
    "rows": int(len(pairs)),
    "anchors_with_candidates": int(pairs["sample_id"].nunique()),
    "anchors_total": int(len(anchors)),
    "benchmark_eligible_anchors_total": int(anchors["primary_evaluation_eligible"].eq("True").sum()),
    "benchmark_eligible_anchors_with_candidates": int(pairs.loc[pairs["sample_id"].isin(set(anchors.loc[anchors["primary_evaluation_eligible"].eq("True"), "sample_id"])), "sample_id"].nunique()),
    "anchors_without_candidates": sorted(all_anchor_ids - pair_anchor_ids),
    "candidate_count_summary": {str(k): float(v) for k, v in pairs.groupby("sample_id").size().describe().items()},
}
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

display(pd.DataFrame([
    ["Candidate pairs", f"{summary['rows']:,}"],
    ["Anchors with candidates", f"{summary['anchors_with_candidates']:,} / {summary['anchors_total']:,}"],
    ["Eligible anchors with candidates", f"{summary['benchmark_eligible_anchors_with_candidates']:,} / {summary['benchmark_eligible_anchors_total']:,}"],
    ["Median candidates per anchor", round(summary['candidate_count_summary'].get('50%', 0), 1)],
], columns=["check", "value"]))
display(pairs.head(10))
assert summary["benchmark_eligible_anchors_with_candidates"] == summary["benchmark_eligible_anchors_total"]
assert pairs["time_gap_days"].min() >= MIN_GAP_DAYS
assert pairs["time_gap_days"].max() <= MAX_GAP_DAYS


## Takeaways

The candidate-pair table now provides a common benchmark input for the baseline linkage algorithms. It is intentionally broad and should be evaluated before any final full-corpus linkage run.
